# parameter-wrap-around-tensor — worked example 3: convert WrapParam then run a fake optimizer step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-wrap-around-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Repairing the composition anti-pattern means converting each `WrapParam` into an `IsAParam` (preserving the backing array) so it survives the isinstance gate. A fake optimizer then updates exactly the surviving MiniTensors.

## Worked solution

We define both designs plus `fix_params`, which maps each `WrapParam` to a new `IsAParam(p.tensor)` and leaves everything else unchanged, returning a fresh list. `fake_optimizer_step(params, lr)` filters to MiniTensors and decrements each survivor's `.array` by `lr` uniformly, returning the count updated. We build a list with a WrapParam and an IsAParam, run the step BEFORE fixing (only one survivor) and AFTER fixing (both survive), and print the two counts to show the repair restores the dropped parameter.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def fix_params(things):
    return [IsAParam(p.tensor) if isinstance(p, WrapParam) else p for p in things]

def fake_optimizer_step(params, lr):
    count = 0
    for p in params:
        if isinstance(p, MiniTensor):
            p.array -= lr * np.ones_like(p.array)
            count += 1
    return count

bag = [WrapParam(np.array([1.0])), IsAParam([2.0])]
print('updated before fix:', fake_optimizer_step(bag, 0.1))   # 1
fixed = fix_params(bag)
print('updated after fix:', fake_optimizer_step(fixed, 0.1))   # 2